In [1]:
# ===================== ALL-IN-ONE: 3D simplex (tetrahedra) for PID-SP =====================
# Requirements: pyomo, gurobi, numpy, scipy, plotly, tqdm, csv file "data.csv"
# -----------------------------------------------------------------------------------------

import numpy as np
import itertools as it
import csv
from tqdm import tqdm
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
from scipy.spatial import Delaunay
import plotly.graph_objects as go
import matplotlib.pyplot as plt

# ===== debug bucket =====
LAST_DEBUG = None   # 如果 next_node == 顶点 或 撞车，会把当时的上下文塞进来

# ------------------------- Config knobs -------------------------
MIN_DIST   = 1e-8     # 去重阈值
ACTIVE_TOL = 1e-8     # active 判定容差
MS_AGG     = "sum"    # 单形 ms 聚合：'sum' 或 'mean'

# ------------------------- PID scenario model -------------------------
def build_pid_model(T=10, h=0.2, scen=None, weights=(1.0, 0.01),
                    bounds=None, use_cvar=False, alpha=0.95):
    assert scen is not None, "请提供一个场景字典"
    Ku, tau, d, sp = scen["Ku"], scen["tau"], scen["d"], scen["sp"]
    assert len(d) == T+1 and len(sp) == T+1

    if bounds is None:
        bounds = {}
    bx = bounds.get("x",  (-20, 20))
    bu = bounds.get("u",  (None, None))
    bKp= bounds.get("Kp", (0, 10))
    bKi= bounds.get("Ki", (0, 10))
    bKd= bounds.get("Kd", (0, 10))
    be = bounds.get("e",  (-100, 100))
    bI = bounds.get("I",  (-200, 200))

    m = pyo.ConcreteModel()
    m.T  = pyo.RangeSet(0, T)
    m.Tm = pyo.RangeSet(1, T)

    m.Kp = pyo.Var(bounds=bKp)
    m.Ki = pyo.Var(bounds=bKi)
    m.Kd = pyo.Var(bounds=bKd)

    m.x = pyo.Var(m.T, bounds=bx)
    m.u = pyo.Var(m.T, bounds=bu)
    m.e = pyo.Var(m.T, bounds=be)
    m.I = pyo.Var(m.T, bounds=bI)

    # error
    def _err_rule(m, t): return m.e[t] == sp[t] - m.x[t]
    m.err_def = pyo.Constraint(m.T, rule=_err_rule)

    # integral
    def _I_dyn(m, t): return m.I[t] == m.I[t-1] + h*m.e[t]
    m.I_dyn = pyo.Constraint(m.Tm, rule=_I_dyn)

    # plant
    def _x_dyn(m, t):
        return m.x[t] == m.x[t-1] + (h/tau)*(-m.x[t] + Ku*m.u[t] + d[t])
    m.x_dyn = pyo.Constraint(m.Tm, rule=_x_dyn)

    # pid
    def _pid_rule(m, t):
        if t == 0:
            return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t]
        return m.u[t] == m.Kp*m.e[t] + m.Ki*m.I[t] + m.Kd*(m.e[t]-m.e[t-1])/h
    m.pid = pyo.Constraint(m.T, rule=_pid_rule)

    m.x0 = pyo.Constraint(expr=m.x[0] == 0)
    m.I0 = pyo.Constraint(expr=m.I[0] == 0)

    w_e, w_u = weights
    m.cost = pyo.Expression(expr=sum(h*(w_e*m.e[t]**2 + w_u*m.u[t]**2) for t in m.T))
    m.obj_expr = pyo.Expression(expr=m.cost)
    return m, [m.Kp, m.Ki, m.Kd]

def load_scenarios_from_csv(csv_path: str, T: int | None = None,
                            sp0: float = 0.0, sp1: float = 0.5,
                            ku_col: str = "tau_us", tau_col: str = "tau_xs",
                            disturb_prefix: str = "disturbance_",
                            setpoint_change_col: str = "setpoint_change"):
    scens = []
    # 推断 T
    if T is None:
        with open(csv_path, "r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            fields  = reader.fieldnames or []
            max_idx = -1
            for name in fields:
                if name.startswith(disturb_prefix):
                    try:
                        k = int(name[len(disturb_prefix):])
                        max_idx = max(max_idx, k)
                    except:
                        pass
            if max_idx < 0:
                raise ValueError(f"未找到扰动列前缀 {disturb_prefix}k")
            T = max_idx

    with open(csv_path, "r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            Ku  = float(row[ku_col])
            tau = float(row[tau_col])

            d = []
            for t in range(T+1):
                col = f"{disturb_prefix}{t}"
                d.append(float(row[col]))

            sp = [sp1]*(T+1)
            if setpoint_change_col in row and row[setpoint_change_col] != "":
                try:
                    t_star = int(float(row[setpoint_change_col]))
                    for t in range(T+1):
                        sp[t] = sp0 if t < t_star else sp1
                except:
                    pass

            scens.append({"Ku": Ku, "tau": tau, "d": d, "sp": sp})
    return scens, T

def build_models_from_csv(csv_path: str, h: float = 0.2,
                          weights=(1.0, 0.01), bounds=None,
                          sp0: float = 0.0, sp1: float = 0.5,
                          ku_col: str = "tau_us", tau_col: str = "tau_xs",
                          disturb_prefix: str = "disturbance_",
                          setpoint_change_col: str = "setpoint_change",
                          max_scenarios=None, skip=0):
    scens, T = load_scenarios_from_csv(
        csv_path=csv_path, T=None, sp0=sp0, sp1=sp1,
        ku_col=ku_col, tau_col=tau_col,
        disturb_prefix=disturb_prefix, setpoint_change_col=setpoint_change_col,
    )
    if skip or max_scenarios:
        scens = scens[skip: (skip + max_scenarios) if max_scenarios else None]

    model_list, first_stg_vars_list = [], []
    for scen in scens:
        m, yvars = build_pid_model(T=T, h=h, scen=scen, weights=weights, bounds=bounds)
        model_list.append(m)
        first_stg_vars_list.append(yvars)

    m_tmpl_list = [model_list[0], first_stg_vars_list[0]]
    return model_list, first_stg_vars_list, m_tmpl_list, T

# ------------------------- Basic utils -------------------------
def corners_from_var_bounds(vars_3):
    bnds = []
    for v in vars_3:
        lb, ub = v.lb, v.ub
        if lb is None or ub is None:
            raise ValueError(f"{v.name} 缺少上下界")
        bnds.append((float(lb), float(ub)))
    return [tuple(p) for p in it.product(*[(lo, hi) for (lo,hi) in bnds])]

def too_close(p, nodes, tol=MIN_DIST):
    return any(np.linalg.norm(np.asarray(p)-np.asarray(q)) < tol for q in nodes)

def evaluate_Q_at(model, first_stg_vars, first_stg_vals, solver):
    """把 Kp,Ki,Kd 固定为给定值，最小化 obj_expr，返回该场景下的真实目标值（用 Gurobi 解）。"""
    # 清理旧目标
    if hasattr(model, 'obj'):
        model.del_component('obj')
    # 固定变量
    for u, v in zip(first_stg_vars, first_stg_vals):
        u.fix(float(v))
    model.obj = pyo.Objective(expr=model.obj_expr, sense=pyo.minimize)
    try:
        results = solver.solve(model, tee=False)
        ok = (results.solver.status == SolverStatus.ok) and \
             (results.solver.termination_condition == TerminationCondition.optimal)
        if not ok:
            raise RuntimeError(f"evaluate_Q_at not optimal: {results.solver.status}, {results.solver.termination_condition}")
        return float(pyo.value(model.obj_expr))
    finally:
        # 清理+解锁
        if hasattr(model, 'obj'):
            model.del_component('obj')
        for u in first_stg_vars:
            if u.fixed:
                u.unfix()


def tet_volume(verts):
    """几何体积（>=0）。verts: 长度4的[(x,y,z),...]"""
    V = np.array(verts, float)
    v0, v1, v2, v3 = V
    return float(abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0)

def tet_quality(verts):
    """
    形状质量分数 q ∈ (0, ~1]，越大越好；sliver 越小。
    定义：q = 6*Vol / sum(edge_len^3)   （无量纲，尺度稳健）
    """
    V = np.array(verts, float)
    # 6 条边
    edges = [np.linalg.norm(V[i] - V[j]) for (i, j) in it.combinations(range(4), 2)]
    denom = float(np.sum(np.power(edges, 3))) + 1e-16
    vol = tet_volume(verts)
    return float(6.0 * vol / denom)

# ------------------------- Single tetra & scene: ms solve -------------------------
def ms_on_tetra_for_scene(model_tmpl, first_vars, solver, tet_vertices, fverts_scene):
    """
    在一个四面体（四个顶点 tet_vertices）上，对单个场景：
      ms = min_{lambda>=0, 1^T lambda=1} [ obj_expr(K(lambda)) - sum_j lambda_j f(v_j) ]
    其中 K(lambda) = sum_j lambda_j * v_j， f(v_j) 是该场景在顶点 v_j 的真实值（已给）。
    返回 (ms_value, lambda*, new_point)；若失败返回 (inf, None, None)

      - A) 对 (顶点, 顶点值) 做同步排序，保证顺序稳定一致
      - B) 接受 locallyOptimal 作为可用解
      - （已去除）向内扰动：直接用 lam* 生成候选点
    """
    # ---- A) 稳定对齐：同步排序 ----
    pairs = sorted(
        [(tuple(map(float, tet_vertices[j])), float(fverts_scene[j])) for j in range(4)],
        key=lambda kv: (kv[0][0], kv[0][1], kv[0][2])
    )
    tet_vertices = [kv[0] for kv in pairs]
    fverts_scene = [kv[1] for kv in pairs]

    # ---- 克隆模型 & 取 Kp,Ki,Kd ----
    m = model_tmpl.clone()
    Kp = m.find_component(first_vars[0].name)
    Ki = m.find_component(first_vars[1].name)
    Kd = m.find_component(first_vars[2].name)
    if any(v is None for v in (Kp, Ki, Kd)):
        raise RuntimeError("克隆模型中找不到 Kp/Ki/Kd")

    # ---- 重心变量 lam ----
    m.lam = pyo.Var(range(4), domain=pyo.NonNegativeReals)
    m.lam_sum = pyo.Constraint(expr=sum(m.lam[j] for j in range(4)) == 1.0)

    vx = [tet_vertices[j][0] for j in range(4)]
    vy = [tet_vertices[j][1] for j in range(4)]
    vz = [tet_vertices[j][2] for j in range(4)]
    m.link_kp = pyo.Constraint(expr=Kp == sum(m.lam[j]*vx[j] for j in range(4)))
    m.link_ki = pyo.Constraint(expr=Ki == sum(m.lam[j]*vy[j] for j in range(4)))
    m.link_kd = pyo.Constraint(expr=Kd == sum(m.lam[j]*vz[j] for j in range(4)))

    # ---- As（该场景）----
    m.As = pyo.Var()
    m.As_def = pyo.Constraint(expr=m.As == sum(m.lam[j]*fverts_scene[j] for j in range(4)))

    # ---- 目标 ----
    if hasattr(m, 'obj'):
        m.del_component('obj')
    m.obj = pyo.Objective(expr=m.obj_expr - m.As, sense=pyo.minimize)

    # ---- 求解（B：接受 locallyOptimal）----
    res = solver.solve(m, tee=False)
    ok = (res.solver.status == SolverStatus.ok) and \
         (res.solver.termination_condition in {
             TerminationCondition.optimal,
             TerminationCondition.locallyOptimal
         })
    if not ok:
        return float('inf'), None, None

    ms_val = float(pyo.value(m.obj))
    lam_star = np.array([pyo.value(m.lam[j]) for j in range(4)], dtype=float)

    # ---- 直接用 lam* 生成候选点（不做任何 ε-padding） ----
    new_pt = np.dot(lam_star, np.array(tet_vertices, dtype=float))
    return ms_val, lam_star, tuple(map(float, new_pt))

# ------------------------- Evaluate all tetrahedra -------------------------
def evaluate_all_tetra(nodes, scen_values, model_list, first_vars_list, solver):
    """
    nodes           : 当前节点列表（3D 点）
    scen_values     : 形状 S×len(nodes)，每个场景在每个节点上的真实值
    返回 tri, per_tet； per_tet 每项含：
      'simplex_index','vert_idx','verts','fverts_sum','ms_per_scene','ms','LB','UB','x_ms_best_scene','best_scene','volume'
      - D) 过滤 sliver 四面体（体积过小）
    """
    pts = np.asarray(nodes, dtype=float)
    if len(pts) < 4:
        return None, []
    tri = Delaunay(pts)  # simplices: (M,4)
    S = len(model_list)

    # 定义域尺度（用于体积阈值）
    mins = pts.min(axis=0)
    maxs = pts.max(axis=0)
    diam = float(np.linalg.norm(maxs - mins))

    vol_tol = 1e-12 * max(diam**3, 1.0)  # 体积阈值（保守值）

    per_tet = []
    for k, simp in enumerate(tri.simplices):
        idxs = list(map(int, simp))
        verts = [tuple(pts[i]) for i in idxs]

        # --- 计算四面体体积 ---
        v0, v1, v2, v3 = np.array(verts)
        vol = abs(np.linalg.det(np.stack([v1 - v0, v2 - v0, v3 - v0], axis=1))) / 6.0

        # ---- D) 过滤 sliver ----
        if vol < vol_tol:
            continue

        # 各场景四顶点值
        fverts_per_scene = [[scen_values[ω][i] for i in idxs] for ω in range(S)]
        # “总 f” 顶点（用于 min/max）
        fverts_sum = [sum(fverts_per_scene[ω][j] for ω in range(S)) for j in range(4)]

        ms_scene = []
        xms_scene = []
        for ω in range(S):
            ms_val, lam_star, new_pt = ms_on_tetra_for_scene(
                model_list[ω], first_vars_list[ω], solver, verts, fverts_per_scene[ω]
            )
            ms_scene.append(ms_val)
            xms_scene.append(new_pt)

        if MS_AGG == "sum":
            ms_total = float(np.sum(ms_scene))
        elif MS_AGG == "mean":
            ms_total = float(np.mean(ms_scene))
        else:
            raise ValueError("MS_AGG must be 'sum' or 'mean'")

        LB = float(np.min(fverts_sum) + ms_total)
        UB = float(np.max(fverts_sum) + ms_total)

        best_scene = int(np.argmin(ms_scene))
        x_ms_best = xms_scene[best_scene]

        per_tet.append({
            "simplex_index": k,
            "vert_idx": idxs,
            "verts": verts,
            "fverts_sum": fverts_sum,
            "ms_per_scene": ms_scene,
            "ms": ms_total,
            "LB": LB,
            "UB": UB,
            "x_ms_best_scene": x_ms_best,
            "best_scene": best_scene,
            "volume": vol,
        })

    return tri, per_tet

# ------------------------- Pretty print -------------------------
def print_tetra_table(per_tet, active_mask, purple_set=None, prec=6):
    purple_set = set() if purple_set is None else set(purple_set)
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    tet_ids = [r["simplex_index"] for r in per_tet]
    active_set = {tid for tid in tet_ids if active_mask.get(tid, False)}

    # 表头：active 加 *，包含最小点加 ^（两者都有 → *^）
    def _mark(tid):
        s = f"T{tid}"
        flags = []
        if tid in active_set:  flags.append("*")
        if tid in purple_set:  flags.append("^")
        return s + ("".join(flags) if flags else "")

    header = ["row\\simp"] + [_mark(tid) for tid in tet_ids]
    rows = [
        ["UB"] + [f"{r['UB']:.{prec}f}" for r in per_tet],
        ["LB"] + [f"{r['LB']:.{prec}f}" for r in per_tet],
        ["ms"] + [f"{r['ms']:.3e}"       for r in per_tet],
    ]

    # 计算列宽
    table = [header] + rows
    colw = [max(len(str(row[c])) for row in table) + 2 for c in range(len(header))]

    # 颜色：active=红；包含最小点=紫；若两者都中，紫优先
    RED, PURPLE, RESET = "\033[31m", "\033[35m", "\033[0m"
    def colorize(col_idx, s):
        if col_idx == 0:
            return s
        tid = tet_ids[col_idx-1]
        if tid in purple_set:
            return f"{PURPLE}{s}{RESET}"
        elif tid in active_set:
            return f"{RED}{s}{RESET}"
        return s

    print("\n== Per-tetra summary ==")
    print("".join(colorize(c, str(header[c]).ljust(colw[c])) for c in range(len(header))))
    print("-"*sum(colw))
    for r in rows:
        line = []
        for c in range(len(header)):
            cell = str(r[c])
            pad  = cell.ljust(colw[c]) if c==0 else cell.rjust(colw[c])
            line.append(colorize(c, pad))
        print("".join(line))
    print("(红色列=active；紫色列=包含当前最小节点的单形；第1行=UB，第2行=LB，第3行=ms)\n")

def min_dist_to_nodes(pt, nodes):
    """最近距离：新点 pt 到历史所有节点 nodes 的最小欧氏距离。"""
    P = np.asarray(pt, float)
    X = np.asarray(nodes, float)
    return float(np.min(np.linalg.norm(X - P, axis=1)))

def print_per_scenario_ms(per_tet, max_scenarios_to_print=10, prec=3):
    per_tet = sorted(per_tet, key=lambda r: r["simplex_index"])
    if not per_tet or "ms_per_scene" not in per_tet[0]:
        return
    S = len(per_tet[0]["ms_per_scene"])
    show = min(S, max_scenarios_to_print)
    head = "simp | " + " ".join([f"s{j}".rjust(10) for j in range(show)])
    print("== Per-tetra per-scenario ms (showing first", show, "of", S, "scenes) ==")
    print(head)
    print("-"*len(head))
    for r in per_tet:
        arr = r["ms_per_scene"][:show]
        sline = " ".join([f"{v:.{prec}e}".rjust(10) for v in arr])
        print(f"{r['simplex_index']:>4d} | {sline}")
    if show < S:
        print(f"... ({S-show} scenes omitted)")
    print()

# ------------------------- Plotly visualization -------------------------
def plot_iteration_plotly(iter_id, nodes, tri, active_mask, ub_node, next_node, per_tet,
                          highlight_simplices=None):
    """
    交互式 3D 可视化：
      - 全部历史节点：黑点
      - 当前最优 UB 节点：绿色圆点
      - 本轮 next node：蓝色菱形
      - 所有 active 单形：半透明橙色四面体 + 深橙色边线
        * 若 next node ≈ 该单形的 x_ms_best_scene，则用更亮的橙色突出
        * 若 highlight_simplices 含该单形 ID，也额外突出
      - 质心 hover：显示 simplex 指标（LB/UB/ms/volume/quality 可选）
    """
    import numpy as np
    import plotly.graph_objects as go

    # 规范化 highlight 集
    if highlight_simplices is None:
        highlight_simplices = set()
    else:
        highlight_simplices = set(highlight_simplices)

    fig = go.Figure()
    nodes = np.asarray(nodes, float)

    # --- 所有节点：黑点 ---
    if len(nodes) > 0:
        fig.add_trace(go.Scatter3d(
            x=nodes[:, 0], y=nodes[:, 1], z=nodes[:, 2],
            mode='markers',
            marker=dict(size=4, color="black"),
            name='nodes'
        ))

    # --- 绿色点：当前最小值节点（UB 节点） ---
    if ub_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[ub_node[0]], y=[ub_node[1]], z=[ub_node[2]],
            mode='markers',
            marker=dict(size=7, symbol="circle", color="green"),
            name='current min node'
        ))

    # --- 蓝色菱形：本轮的 next node ---
    if next_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[next_node[0]], y=[next_node[1]], z=[next_node[2]],
            mode='markers',
            marker=dict(size=8, symbol="diamond", color="#1976d2"),
            name='next node'
        ))

    # --- active 单形：橙色半透明四面体 + 深橙色边线 + hover 质心 ---
    def _is_same_point(a, b, atol=1e-6):
        if a is None or b is None:
            return False
        return np.linalg.norm(np.asarray(a, float) - np.asarray(b, float)) <= float(atol)

    if tri is not None:
        legend_mesh_added = False
        legend_edge_added = False

        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue

            verts = np.array(r["verts"], dtype=float)  # (4,3)

            # 是否突出：1) next_node ≈ x_ms_best_scene；或 2) sid 在 highlight_simplices
            highlight_by_next = _is_same_point(next_node, r.get("x_ms_best_scene", None), atol=1e-6)
            highlight = highlight_by_next or (sid in highlight_simplices)

            mesh_color = "#ff5722" if highlight else "#ffb74d"   # 高亮橙 vs 普通橙
            edge_color = "darkorange"
            edge_width = 4 if highlight else 3
            mesh_opacity = 0.45 if highlight else 0.35

            # 四面体四个三角面
            I = [0, 0, 0, 1]
            J = [1, 1, 2, 2]
            K = [2, 3, 3, 3]

            # mesh
            fig.add_trace(go.Mesh3d(
                x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
                i=I, j=J, k=K,
                color=mesh_color,
                opacity=mesh_opacity,
                showscale=False,
                name="active simplex",
                showlegend=(not legend_mesh_added)
            ))
            legend_mesh_added = True

            # 边线
            edges = [(0,1), (0,2), (0,3), (1,2), (1,3), (2,3)]
            for (a, b) in edges:
                pa, pb = verts[a], verts[b]
                fig.add_trace(go.Scatter3d(
                    x=[pa[0], pb[0]],
                    y=[pa[1], pb[1]],
                    z=[pa[2], pb[2]],
                    mode='lines',
                    line=dict(width=edge_width, color=edge_color),
                    name='active edge',
                    showlegend=(not legend_edge_added)
                ))
            legend_edge_added = True

            # 质心 hover：展示 sid / LB / UB / ms / volume (/quality)
            cx, cy, cz = np.mean(verts, axis=0)
            qtxt = ""
            if "quality" in r and r["quality"] is not None:
                try:
                    qtxt = f"<br>q={float(r['quality']):.3e}"
                except Exception:
                    qtxt = ""
            txt = (f"simp={sid}"
                   f"<br>LB={float(r['LB']):.6f}"
                   f"<br>UB={float(r['UB']):.6f}"
                   f"<br>ms={float(r['ms']):.3e}"
                   f"<br>vol={float(r['volume']):.3e}"
                   f"{qtxt}")

            # 用透明的小点承载 hover 文本（不加入图例）
            fig.add_trace(go.Scatter3d(
                x=[cx], y=[cy], z=[cz],
                mode='markers',
                marker=dict(size=1, opacity=0.0),
                text=[txt], hoverinfo="text",
                name="tetra info",
                showlegend=False
            ))

    # --- 布局 ---
    fig.update_layout(
        title=f"Iteration {iter_id}",
        scene=dict(
            xaxis_title="Kp",
            yaxis_title="Ki",
            zaxis_title="Kd",
            aspectmode="cube",
            zaxis=dict(tickformat=".2f"),  # 减少"e/p"渲染问题
        ),
        width=980,
        height=720,
        legend=dict(itemsizing="constant")
    )

    # 让 hover 更干净（避免科学计数法误渲染）
    fig.update_traces(
        hovertemplate="x: %{x:.6f}<br>y: %{y:.6f}<br>z: %{z:.6f}",
        selector=dict(type='scatter3d')
    )

    fig.show()

def run_pid_simplex_3d(model_list, first_vars_list, solver, target_nodes=30,
                       min_dist=MIN_DIST, active_tol=ACTIVE_TOL, verbose=True):
    global LAST_DEBUG
    # 历史记录
    LB_hist, UB_hist, ms_hist, node_count = [], [], [], []
    UB_node_hist, add_node_hist = [], []

    # 新增的历史
    ms_a_hist, ms_b_hist = [], []
    active_ratio_hist = []

    S = len(model_list)

    # 初始节点：变量上下界的角点
    nodes = corners_from_var_bounds(first_vars_list[0])

    # C.0 保持用户传入的 min_dist（不放大）
    bounds_arr = np.array([[float(v.lb), float(v.ub)] for v in first_vars_list[0]], float)
    diam = float(np.linalg.norm(bounds_arr[:,1] - bounds_arr[:,0]))
    min_dist = float(min_dist)

    # 缓存 f_ω(node_i)
    scen_values = [[None]*len(nodes) for _ in range(S)]
    for i, node in enumerate(nodes):
        for ω in range(S):
            scen_values[ω][i] = evaluate_Q_at(model_list[ω], first_vars_list[ω], node, solver)

    it = 0
    stop_due_to_collision = False
    while len(nodes) < target_nodes:
        # 1) 全局 UB
        f_sum_per_node = [
            sum(scen_values[ω][i] for ω in range(S))
            for i in range(len(nodes))
        ]
        ub_idx = int(np.argmin(f_sum_per_node))
        UB_global = float(f_sum_per_node[ub_idx])
        UB_node = tuple(nodes[ub_idx])

        # 2) 评估所有四面体
        tri, per_tet = evaluate_all_tetra(
            nodes, scen_values, model_list, first_vars_list, solver
        )
        if tri is None or not per_tet:
            if verbose:
                print("Not enough nodes to make tetrahedra; stop.")
            break

        # 3) active mask（先按 UB 过滤）
        active_mask = {
            r["simplex_index"]: (r["LB"] <= UB_global + active_tol)
            for r in per_tet
        }
        # 形状质量过滤
        q_cut = 1e-3   # 你可以按需调整
        for r in per_tet:
            sid = r["simplex_index"]
            if not active_mask.get(sid, False):
                continue
            q = tet_quality(r["verts"])
            if q < q_cut:
                active_mask[sid] = False

        # 4) active ratio
        total_vol = sum(r["volume"] for r in per_tet)
        active_vol = sum(r["volume"] for r in per_tet if active_mask[r["simplex_index"]])
        active_ratio = active_vol / total_vol if total_vol > 0 else 0.0

        # 5) LB_global & ms_b & 其来源单形编号 —— 只在 active ∩ 含 UB 节点 中取
        ub_active = [r for r in per_tet
                    if (ub_idx in r["vert_idx"]) and active_mask.get(r["simplex_index"], False)]

        if ub_active:
            ms_b_rec   = min(ub_active, key=lambda r: r["ms"])
            ms_b       = float(ms_b_rec["ms"])
            ms_b_simp  = int(ms_b_rec["simplex_index"])
            LB_global  = UB_global + ms_b
        else:
            # 若 UB 邻域没有 active 单形，则退回“所有 active”的最小 LB（再退全体）
            ms_b       = float('nan')
            ms_b_simp  = None
            active_LBs = [r["LB"] for r in per_tet if active_mask.get(r["simplex_index"], False)]
            LB_global  = float(min(active_LBs)) if active_LBs else float(min(r["LB"] for r in per_tet))


        # 6) ms_a（active 内最小 ms）
        if any(active_mask.values()):
            ms_a = float(min(r["ms"] for r in per_tet if active_mask[r["simplex_index"]]))
        else:
            ms_a = float(min(r["ms"] for r in per_tet))
        ms_iter = ms_a

        # 7) 记录到历史
        LB_hist.append(LB_global)
        UB_hist.append(UB_global)
        ms_hist.append(ms_iter)
        node_count.append(len(nodes))
        UB_node_hist.append(UB_node)
        ms_a_hist.append(ms_a)
        ms_b_hist.append(ms_b)
        active_ratio_hist.append(active_ratio)

        # 8) 打印（含 LB、ms_b、ms_b 来源单形）
        simp_with_min = [r["simplex_index"] for r in per_tet if ub_idx in r["vert_idx"]]
        if verbose:
            print(f"[Iter {it}] Active simplex ratio = {active_ratio:.6f}")
            print(f"[Iter {it}] UB node {UB_node} is in simplices {sorted(simp_with_min)}")
            msb_src = f"T{ms_b_simp}" if ms_b_simp is not None else "N/A"
            print(f"[Iter {it}] LB = {LB_global:.6f} = UB({UB_global:.6f}) + ms_b({ms_b:.3e}) from {msb_src}")

        # 9) 候选排行：优先在“包含 UB 节点”的 active 单形中选点（若没有则退回全部 active）
        active = [r for r in per_tet if active_mask[r["simplex_index"]]]
        ub_active = [r for r in active if ub_idx in r["vert_idx"]]
        candidates_pool = ub_active if len(ub_active) > 0 else active

        def score(cand):
            ms = cand["ms"]
            pt = cand.get("x_ms_best_scene", None)
            d  = (float('inf') if pt is None else min_dist_to_nodes(pt, nodes))
            return (ms, -d)

        candidates_sorted = sorted(candidates_pool, key=score)
        # 候选第一名（若存在），用于和 ms_b 对照
        if verbose:
            top_msg = "N/A"
            if len(candidates_sorted) > 0:
                top0 = candidates_sorted[0]
                top_msg = f"T{int(top0['simplex_index'])}, ms={float(top0['ms']):.3e}"
            msb_src = f"T{ms_b_simp}" if ms_b_simp is not None else "N/A"
            print(f"[Iter {it}] LB = {LB_global:.6f} = UB({UB_global:.6f}) + ms_b({ms_b:.3e}) from {msb_src}")
            print(f"[Iter {it}] candidate rank #1: {top_msg}")


        if verbose:
            topN = candidates_sorted[:10]
            print("== ms candidates (sorted by (ms, -dist)) ==")
            print(f"{'rank':>4} {'simp':>6} {'ms':>12} {'mind(all)':>12} {'pt':>30}")
            print("-" * 90)
            for rnk, cand in enumerate(topN, start=1):
                pt = cand.get("x_ms_best_scene", None)
                d  = (float('nan') if pt is None else min_dist_to_nodes(pt, nodes))
                pt_str = "None" if pt is None else f"({pt[0]:.4f}, {pt[1]:.4f}, {pt[2]:.4f})"
                print(f"{rnk:>4} T{cand['simplex_index']:<4} {cand['ms']:>12.4e} {d:>12.2e} {pt_str:>30}")
            print()

        # 10) 选新点 + 强校验/撞车处理
        new_node = None
        chosen_ms = None
        chosen_cand = None
        stop_due_to_collision = False

        def handle_collision(cand_pt, cand, stage_note="active"):
            """处理撞车：打包 LAST_DEBUG；可视化高亮；停止迭代。"""
            nonlocal stop_due_to_collision
            # 找最近已存在点
            X = np.asarray(nodes, float)
            P = np.asarray(cand_pt, float)
            dists = np.linalg.norm(X - P, axis=1)
            j_star = int(np.argmin(dists))
            d_star = float(dists[j_star])

            # 找出所有包含该已存在点的单形
            orange_ids = [r["simplex_index"] for r in per_tet if j_star in r["vert_idx"]]

            # 打包快照
            debug_pack = {
                "reason": "candidate_too_close",
                "iter": it,
                "stage": stage_note,
                "min_dist": float(min_dist),
                "closest_node_index": j_star,
                "closest_node_point": tuple(map(float, nodes[j_star])),
                "closest_distance": d_star,
                "cand_simplex": int(cand["simplex_index"]),
                "cand_point": tuple(map(float, cand_pt)),
                "cand_ms": float(cand["ms"]),
                "UB_global": float(UB_global),
                "LB_global": float(LB_global),
                "active_ratio": float(active_ratio),
                "UB_node": tuple(map(float, UB_node)),
                "active_mask": {int(k): bool(v) for k, v in active_mask.items()},
                "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
                "per_tet_snapshot": [
                    {
                        "simplex_index": int(r["simplex_index"]),
                        "vert_idx": list(map(int, r["vert_idx"])),
                        "verts": [tuple(map(float, x)) for x in r["verts"]],
                        "ms": float(r["ms"]),
                        "LB": float(r["LB"]),
                        "UB": float(r["UB"]),
                        "best_scene": int(r["best_scene"]),
                        "x_ms_best_scene": tuple(map(float, r["x_ms_best_scene"])),
                        "volume": float(r["volume"]),
                    } for r in per_tet
                ],
                "highlight_simplices": list(map(int, orange_ids)),
            }
            # 写入 LAST_DEBUG
            global LAST_DEBUG
            LAST_DEBUG = debug_pack

            # 可视化：把撞车点作为 next_node 传入，并高亮相关单形
            plot_iteration_plotly(
                it, nodes, tri, active_mask, UB_node, cand_pt, per_tet,
                highlight_simplices=orange_ids
            )

            # 打印提示并停
            if verbose:
                print(
                    f"[STOP] Candidate {tuple(map(float, cand_pt))} "
                    f"is too close to existing node #{j_star} at distance {d_star:.3e} "
                    f"(< {min_dist:g}). Highlighted simplices: {sorted(orange_ids)}"
                )
            stop_due_to_collision = True

        # 主候选循环
        for rank, cand in enumerate(candidates_sorted, start=1):
            cand_pt = cand.get("x_ms_best_scene", None)
            if cand_pt is None:
                continue
            if min_dist_to_nodes(cand_pt, nodes) >= min_dist:
                new_node   = cand_pt
                chosen_ms  = cand["ms"]
                chosen_cand= cand
                if verbose:
                    print(
                        f"Chosen node {tuple(map(float, cand_pt))} "
                        f"with ms={chosen_ms:.3e} "
                        f"(simp T{cand['simplex_index']}, rank #{rank})"
                    )
                    # 打印 next node 来源单形
                    print(f"[Iter {it}] next node comes from simplex T{int(cand['simplex_index'])}")
                break
            else:
                if verbose:
                    print(
                        f"Skip candidate {tuple(map(float, cand_pt))} "
                        f"(simp T{cand['simplex_index']}, rank #{rank}) "
                        f"because too close to existing nodes (< {min_dist:g})."
                    )
                # 撞车：高亮并停止（不抛异常）
                handle_collision(cand_pt, cand, stage_note="active")
                break  # 结束候选循环

        # fallback：如果没选中且没因撞车停止，再用全体单形试一次
        if (new_node is None) and (not stop_due_to_collision) and (len(active) > 0):
            if verbose:
                print("[fallback] All active candidates too close; try all simplices...")
            all_sorted = sorted(per_tet, key=score)
            for cand in all_sorted:
                cand_pt = cand.get("x_ms_best_scene", None)
                if cand_pt is None:
                    continue
                if min_dist_to_nodes(cand_pt, nodes) >= min_dist:
                    new_node   = cand_pt
                    chosen_ms  = cand["ms"]
                    chosen_cand= cand
                    if verbose:
                        print(
                            f"Chosen node {tuple(map(float, cand_pt))} "
                            f"with ms={chosen_ms:.3e} "
                            f"(simp T{cand['simplex_index']}) [fallback-all]"
                        )
                        print(f"[Iter {it}] next node comes from simplex T{int(cand['simplex_index'])} [fallback-all]")
                    break
                else:
                    if verbose:
                        print(
                            f"Skip (all) candidate {tuple(map(float, cand_pt))} "
                            f"(simp T{cand['simplex_index']}) "
                            f"because too close to existing nodes (< {min_dist:g})."
                        )
                    handle_collision(cand_pt, cand, stage_note="fallback-all")
                    break  # 结束 fallback 循环

        # 若发生撞车，则停止整个 while
        if stop_due_to_collision:
            if verbose:
                print(f"[Iter {it}] Stop due to collision.")
            break

        # 如果依然没找到新点，就停
        if new_node is None:
            if verbose:
                print("New node too close for all candidates (or infeasible ms); stop.")
            break

        # == 强校验：是否与候选单形某个顶点相同？若相同则打包并停止 ==
        tol_same = 1e-10
        def _same(a, b, tol=tol_same):
            a = np.asarray(a, float); b = np.asarray(b, float)
            return np.linalg.norm(a - b) <= tol

        if chosen_cand is not None:
            offending_vert = None
            for v in chosen_cand["verts"]:
                if _same(new_node, v):
                    offending_vert = tuple(map(float, v))
                    break
            if offending_vert is not None:
                LAST_DEBUG = {
                    "reason": "next_node_equals_vertex",
                    "iter": it,
                    "new_node": tuple(map(float, new_node)),
                    "offending_vertex": offending_vert,
                    "tol_same": tol_same,
                    "candidate": {
                        "simplex_index": int(chosen_cand["simplex_index"]),
                        "vert_idx": list(map(int, chosen_cand["vert_idx"])),
                        "verts": [tuple(map(float, x)) for x in chosen_cand["verts"]],
                        "ms": float(chosen_cand["ms"]),
                        "ms_per_scene": [float(x) for x in chosen_cand["ms_per_scene"]],
                        "best_scene": int(chosen_cand["best_scene"]),
                        "x_ms_best_scene": tuple(map(float, chosen_cand["x_ms_best_scene"])),
                        "LB": float(chosen_cand["LB"]),
                        "UB": float(chosen_cand["UB"]),
                        "volume": float(chosen_cand["volume"]),
                    },
                    "UB_global": float(UB_global),
                    "LB_global": float(LB_global),
                    "active_ratio": float(active_ratio),
                    "UB_node": tuple(map(float, UB_node)),
                    "active_mask": {int(k): bool(v) for k, v in active_mask.items()},
                    "nodes_snapshot": [tuple(map(float, nd)) for nd in nodes],
                    "per_tet_snapshot": [
                        {
                            "simplex_index": int(r["simplex_index"]),
                            "vert_idx": list(map(int, r["vert_idx"])),
                            "verts": [tuple(map(float, x)) for x in r["verts"]],
                            "ms": float(r["ms"]),
                            "LB": float(r["LB"]),
                            "UB": float(r["UB"]),
                            "best_scene": int(r["best_scene"]),
                            "x_ms_best_scene": tuple(map(float, r["x_ms_best_scene"])),
                            "volume": float(r["volume"]),
                        } for r in per_tet
                    ],
                }
                # 高亮包含该 offending_vert 的单形
                vert_idx_list = []
                for j, nd in enumerate(nodes):
                    if _same(offending_vert, nd):
                        vert_idx_list.append(j)
                orange_ids = [r["simplex_index"] for r in per_tet if any(j in r["vert_idx"] for j in vert_idx_list)]
                plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet,
                                      highlight_simplices=orange_ids)
                if verbose:
                    print("[STOP] new_node coincides with a simplex vertex. Highlighted simplices:",
                          sorted(orange_ids))
                break  # 停止

        # 可视化（正常迭代：不着色任何单形）
        plot_iteration_plotly(it, nodes, tri, active_mask, UB_node, new_node, per_tet,
                              highlight_simplices=None)

        # 加点并评估
        new_vals = []
        for ω in range(S):
            val = evaluate_Q_at(model_list[ω], first_vars_list[ω], new_node, solver)
            new_vals.append(val)

        nodes.append(tuple(map(float, new_node)))
        for ω in range(S):
            scen_values[ω].append(new_vals[ω])

        add_node_hist.append(new_node)
        it += 1  # 下一轮

    # while 结束

    return {
        "nodes": np.array(nodes, float),
        "LB_hist": LB_hist,
        "UB_hist": UB_hist,
        "ms_hist": ms_hist,               # = ms_a
        "ms_a_hist": ms_a_hist,           # active 最小 ms
        "ms_b_hist": ms_b_hist,           # 用于 LB 的 ms（UB节点邻域里 min）
        "node_count": node_count,
        "UB_node_hist": UB_node_hist,
        "added_nodes": add_node_hist,
        "active_ratio_hist": active_ratio_hist,
    }



# ===================== MAIN =====================
RUN_QUICK_TEST = True  # True: 先用小规模验证

if RUN_QUICK_TEST:
    csv_path       = "data.csv"
    max_scenarios  = 1
    target_nodes   = 30
else:
    csv_path       = "data.csv"
    max_scenarios  = 99
    target_nodes   = 30

bounds = {
    "x":  (None, None),
    "u":  (None, None),
    "e":  (None, None),
    "I":  (-10, 10),
    "Kp": (0, 1),
    "Ki": (0, 1),
    "Kd": (0, 1),
}
weights = (1.0, 0.01)

# build models
model_list, first_stg_vars_list, m_tmpl_list, T = build_models_from_csv(
    csv_path, h=0.2, weights=weights, bounds=bounds,
    sp0=0.0, sp1=0.5, ku_col="tau_us", tau_col="tau_xs",
    disturb_prefix="disturbance_", setpoint_change_col="setpoint_change",
    max_scenarios=max_scenarios, skip=0
)

# Gurobi solver
solver = pyo.SolverFactory('gurobi')
solver.options.update({
    'MIPGap': 1e-1,
    'NumericFocus': 1,
    'Presolve': 2,
    'NonConvex': 2,   # 必须
    'TimeLimit': 10,  # 可按需打开
})

# run
hist = run_pid_simplex_3d(
    model_list=model_list,
    first_vars_list=first_stg_vars_list,
    solver=solver,
    target_nodes=target_nodes,
    min_dist=MIN_DIST,
    active_tol=ACTIVE_TOL,
    verbose=True
)

print("\n==== Done ====")
print(f"Total nodes: {len(hist['nodes'])}")
print(f"Best UB: {min(hist['UB_hist']) if hist['UB_hist'] else None}")
print(f"Last LB: {hist['LB_hist'][-1] if hist['LB_hist'] else None}")
# ===========================================================================================


[Iter 0] Active simplex ratio = 1.000000
[Iter 0] UB node (1.0, 1.0, 1.0) is in simplices [2, 3]
[Iter 0] LB = 0.162061 = UB(0.361626) + ms_b(-1.996e-01) from T2
[Iter 0] LB = 0.162061 = UB(0.361626) + ms_b(-1.996e-01) from T2
[Iter 0] candidate rank #1: T2, ms=-1.996e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T2     -1.9957e-01     5.23e-01       (1.0000, 0.3698, 0.3698)
   2 T3     -1.9957e-01     5.23e-01       (1.0000, 0.3698, 0.3698)

Chosen node (0.9999999820786598, 0.3698004755110178, 0.36980053410465885) with ms=-1.996e-01 (simp T2, rank #1)
[Iter 0] next node comes from simplex T2


[Iter 1] Active simplex ratio = 1.000000
[Iter 1] UB node (1.0, 1.0, 1.0) is in simplices [2, 3, 4, 5, 10, 11]
[Iter 1] LB = -0.558282 = UB(0.361626) + ms_b(-9.199e-01) from T3
[Iter 1] LB = -0.558282 = UB(0.361626) + ms_b(-9.199e-01) from T3
[Iter 1] candidate rank #1: T3, ms=-9.199e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T3     -9.1991e-01     4.41e-01       (0.3117, 0.3117, 1.0000)
   2 T2     -9.1991e-01     4.41e-01       (0.3117, 0.3117, 1.0000)
   3 T5     -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   4 T4     -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)

Chosen node (0.31167947243602057, 0.3116794690303519, 0.9999999984121735) with ms=-9.199e-01 (simp T3, rank #1)
[Iter 1] next node comes from simplex T3


[Iter 2] Active simplex ratio = 0.905630
[Iter 2] UB node (1.0, 1.0, 1.0) is in simplices [4, 5, 6, 7, 10, 11, 16, 17]
[Iter 2] LB = 0.071659 = UB(0.361626) + ms_b(-2.900e-01) from T7
[Iter 2] LB = 0.071659 = UB(0.361626) + ms_b(-2.900e-01) from T7
[Iter 2] candidate rank #1: T7, ms=-2.900e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T7     -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   2 T6     -2.8997e-01     4.54e-01       (0.3214, 1.0000, 0.3214)
   3 T4     -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)
   4 T5     -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)

Chosen node (0.3213541220655616, 0.999999993125058, 0.3213541087355617) with ms=-2.900e-01 (simp T7, rank #1)
[Iter 2] next node comes from simplex T7


[Iter 3] Active simplex ratio = 0.863803
[Iter 3] UB node (1.0, 1.0, 1.0) is in simplices [0, 6, 9, 10, 11, 12, 15, 16, 21, 22]
[Iter 3] LB = 0.222111 = UB(0.361626) + ms_b(-1.395e-01) from T9
[Iter 3] LB = 0.222111 = UB(0.361626) + ms_b(-1.395e-01) from T9
[Iter 3] candidate rank #1: T9, ms=-1.395e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T9     -1.3952e-01     3.72e-01       (0.4631, 0.6153, 0.8478)
   2 T10    -1.3952e-01     3.72e-01       (0.4631, 0.6153, 0.8478)
   3 T6     -1.3590e-01     3.67e-01       (0.5709, 0.5709, 1.0000)
   4 T0     -5.7121e-02     4.20e-01       (1.0000, 0.8296, 0.3836)

Chosen node (0.4631080160042133, 0.6152959121308406, 0.8478121148030885) with ms=-1.395e-01 (simp T9, rank #1)
[Iter 3] next node comes from simplex T9


[Iter 4] Active simplex ratio = 0.826469
[Iter 4] UB node (1.0, 1.0, 1.0) is in simplices [0, 1, 2, 3, 7, 8, 17, 18, 21, 22, 27, 28]
[Iter 4] LB = 0.225722 = UB(0.361626) + ms_b(-1.359e-01) from T7
[Iter 4] LB = 0.225722 = UB(0.361626) + ms_b(-1.359e-01) from T7
[Iter 4] candidate rank #1: T7, ms=-1.359e-01
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T7     -1.3590e-01     1.92e-01       (0.5709, 0.5709, 1.0000)
   2 T8     -1.3590e-01     1.92e-01       (0.5709, 0.5709, 1.0000)
   3 T0     -1.1924e-01     4.02e-01       (1.0000, 0.4023, 1.0000)
   4 T2     -5.7121e-02     4.20e-01       (1.0000, 0.8296, 0.3836)
   5 T3     -5.0312e-02     4.47e-01       (0.6991, 0.8481, 0.5473)
   6 T1     -4.8200e-02     3.85e-01       (0.5506, 0.9283, 0.6223)

Chosen node (0.5709144954540724, 0.5709143802363048, 0.9999999958322288) with 

...

[Iter 18] Active simplex ratio = 0.601163
[Iter 18] UB node (1.0, 1.0, 1.0) is in simplices [6, 7, 14, 16, 20, 21, 22, 24, 25, 38, 39, 40, 50, 52, 54, 55, 56, 68, 69, 75, 76, 77, 78, 79, 93, 94, 101, 102, 103, 104]
[Iter 18] LB = 0.350993 = UB(0.361626) + ms_b(-1.063e-02) from T38
[Iter 18] LB = 0.350993 = UB(0.361626) + ms_b(-1.063e-02) from T38
[Iter 18] candidate rank #1: T38, ms=-1.063e-02
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T38    -1.0633e-02     1.71e-01       (0.8851, 1.0000, 0.6964)
   2 T25    -1.0600e-02     2.55e-01       (0.7324, 0.9292, 0.8216)
   3 T78    -1.0600e-02     2.55e-01       (0.7324, 0.9292, 0.8216)
   4 T40    -1.0390e-02     1.18e-01       (0.9886, 1.0000, 0.6610)
   5 T20    -9.8507e-03     2.53e-01       (0.7702, 0.9445, 0.7876)
   6 T77    -9.8189e-03     2.64e-01       (0.7110, 0.9765,

[Iter 19] Active simplex ratio = 0.586958
[Iter 19] UB node (1.0, 1.0, 1.0) is in simplices [6, 7, 14, 16, 20, 21, 23, 34, 35, 36, 38, 40, 42, 44, 45, 46, 58, 59, 65, 66, 67, 68, 69, 81, 82, 83, 99, 100, 107, 108, 109, 110]
[Iter 19] LB = 0.351241 = UB(0.361626) + ms_b(-1.039e-02) from T34
[Iter 19] LB = 0.351241 = UB(0.361626) + ms_b(-1.039e-02) from T34
[Iter 19] candidate rank #1: T34, ms=-1.039e-02
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T34    -1.0385e-02     1.17e-01       (1.0000, 1.0000, 0.6575)
   2 T83    -6.8982e-03     1.87e-01       (0.7739, 0.9100, 0.9131)
   3 T82    -6.7888e-03     2.09e-01       (0.7384, 0.9374, 0.9289)
   4 T36    -6.6277e-03     1.88e-01       (0.8465, 0.8750, 0.8821)
   5 T35    -6.2327e-03     1.69e-01       (0.9097, 0.8549, 0.8558)
   6 T66    -6.0929e-03     1.81e-01       (0.9407

[Iter 20] Active simplex ratio = 0.586349
[Iter 20] UB node (1.0, 1.0, 1.0) is in simplices [6, 7, 14, 16, 20, 21, 23, 34, 35, 37, 39, 41, 43, 44, 45, 60, 61, 62, 63, 74, 75, 76, 77, 80, 81, 88, 89, 90, 106, 107, 114, 115, 116, 117]
[Iter 20] LB = 0.354728 = UB(0.361626) + ms_b(-6.898e-03) from T90
[Iter 20] LB = 0.354728 = UB(0.361626) + ms_b(-6.898e-03) from T90
[Iter 20] candidate rank #1: T90, ms=-6.898e-03
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T90    -6.8982e-03     1.87e-01       (0.7739, 0.9100, 0.9131)
   2 T89    -6.7888e-03     2.09e-01       (0.7384, 0.9374, 0.9289)
   3 T35    -6.6277e-03     1.88e-01       (0.8465, 0.8750, 0.8821)
   4 T34    -6.2327e-03     1.69e-01       (0.9097, 0.8549, 0.8558)
   5 T61    -6.0929e-03     1.81e-01       (0.9407, 0.8319, 0.8968)
   6 T60    -6.0866e-03     1.78e-01     

[Iter 21] Active simplex ratio = 0.582721
[Iter 21] UB node (1.0, 1.0, 1.0) is in simplices [6, 7, 14, 16, 20, 21, 34, 35, 37, 39, 40, 41, 56, 57, 58, 59, 70, 71, 72, 73, 76, 77, 84, 85, 86, 88, 89, 96, 97, 98, 112, 113, 120, 121, 122, 123]
[Iter 21] LB = 0.355238 = UB(0.361626) + ms_b(-6.389e-03) from T98
[Iter 21] LB = 0.355238 = UB(0.361626) + ms_b(-6.389e-03) from T98
[Iter 21] candidate rank #1: T98, ms=-6.389e-03
== ms candidates (sorted by (ms, -dist)) ==
rank   simp           ms    mind(all)                             pt
------------------------------------------------------------------------------------------
   1 T98    -6.3888e-03     9.21e-02       (0.7478, 0.9256, 1.0000)
   2 T97    -6.3852e-03     9.53e-02       (0.7408, 0.9309, 1.0000)
   3 T57    -6.0929e-03     1.81e-01       (0.9407, 0.8319, 0.8968)
   4 T56    -6.0866e-03     1.78e-01       (0.9410, 0.8335, 0.8881)
   5 T85    -6.0865e-03     1.71e-01       (0.9283, 0.8364, 0.9036)
   6 T88    -6.0720e-03     1.63e


==== Done ====
Total nodes: 30
Best UB: 0.3616264624871053
Last LB: 0.35523768728020017


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ast
import os

# ===== 0) 准备单形法数据（来自 hist）=====
simp = pd.DataFrame({
    "k": np.asarray(hist.get("node_count", []), dtype=float),
    "UB_simplex": np.asarray(hist.get("UB_hist", []), dtype=float),
    "LB_simplex": np.asarray(hist.get("LB_hist", []), dtype=float),
    "ms_a": np.asarray(hist.get("ms_a_hist", []), dtype=float),   # active 单形里最小 ms
    "ms_b": np.asarray(hist.get("ms_b_hist", []), dtype=float),   # 用于算 LB 的 ms（含 UB 节点的单形里 min）
}).sort_values("k").reset_index(drop=True)

# 保护：如果 hist 为空，直接报个友好错误
if simp.empty or simp["k"].isna().all():
    raise RuntimeError("hist 为空或缺少必要字段（node_count/UB_hist/LB_hist）。先运行主算法得到 hist。")

# ===== 1) 读取 & 清洗分片法 CSV（若文件存在则对齐对比）=====
csv_path = "mydata0.csv"
pw = None

def _parse_ms_global(row):
    # 解析 "ms" 列（可能是字符串列表），否则退回 "sum_ms"
    v = row.get("ms", np.nan)
    if isinstance(v, str):
        try:
            arr = ast.literal_eval(v)
            if isinstance(arr, (list, tuple)) and len(arr) > 0:
                return float(np.min(arr))
        except Exception:
            pass
    sm = row.get("sum_ms", np.nan)
    return float(sm) if pd.notna(sm) else np.nan

if os.path.exists(csv_path):
    cols_present = pd.read_csv(csv_path, nrows=0).columns.tolist()
    need_cols = [c for c in ["k", "LB", "UB", "ms", "sum_ms"] if c in cols_present]
    if need_cols:
        pw = pd.read_csv(csv_path, usecols=need_cols)
        # 数值化
        for c in ["k", "LB", "UB", "sum_ms"]:
            if c in pw.columns:
                pw[c] = pd.to_numeric(pw[c], errors="coerce")
        # 解析 ms
        if "ms" in pw.columns or "sum_ms" in pw.columns:
            pw["ms_pw"] = pw.apply(_parse_ms_global, axis=1)
        # 同一 k 只保留“最后一条”
        pw = pw.dropna(subset=["k"]).sort_values("k").groupby("k", as_index=False).last()
        pw["k"] = pw["k"].astype(float)

# ===== 2) 按单形法 k 对齐（<=k 的最近一条）=====
if pw is not None:
    aligned = pd.merge_asof(
        left=simp.sort_values("k"),
        right=pw[["k"] + [c for c in ["LB","UB","ms_pw"] if c in (pw.columns if pw is not None else [])]].sort_values("k"),
        on="k",
        direction="backward"
    )
else:
    aligned = simp.copy()

# ===== 3) 画图：UB/LB vs 节点数 =====
plt.figure(figsize=(7,4.5))
plt.plot(aligned["k"], aligned["UB_simplex"], marker='s', label="Simplex UB")
plt.plot(aligned["k"], aligned["LB_simplex"], marker='o', linestyle="--", label="Simplex LB")
if pw is not None and "UB" in aligned.columns:
    plt.plot(aligned["k"], aligned["UB"], marker='s', label="Piecewise UB")
if pw is not None and "LB" in aligned.columns:
    plt.plot(aligned["k"], aligned["LB"], marker='o', linestyle="--", label="Piecewise LB")
plt.xlabel("Number of nodes")
plt.ylabel("Bound value")
plt.title("UB & LB vs Number of Nodes (Aligned to Simplex)")
plt.grid(True); plt.legend(); plt.tight_layout(); plt.show()

# ===== 4) 画图：ms vs 节点数 =====
plt.figure(figsize=(7,4.5))
plt.plot(aligned["k"], aligned["ms_a"], marker='o', label="Simplex ms_a (min over active)")
plt.plot(aligned["k"], aligned["ms_b"], marker='s', label="Simplex ms_b (LB-driver)")
if pw is not None and "ms_pw" in aligned.columns:
    plt.plot(aligned["k"], aligned["ms_pw"], marker='^', label="Piecewise ms (global min)")
plt.xlabel("Number of nodes")
plt.ylabel("ms")
plt.title("ms vs Number of Nodes (Aligned to Simplex)")
plt.grid(True); plt.legend(); plt.tight_layout(); plt.show()


NameError: name 'hist' is not defined

In [2]:
# =================== Visualize one simplex & color obj - As on a barycentric grid ===================

import numpy as np
import plotly.graph_objects as go

# 配置：重心网格的细密程度（n 越大点越多；点数约为 C(n+3,3)）
N_BARY = 8  # 建议 6~12 之间，过大计算会慢（每个点都要解一次模型）

def _fetch_problem_simplex_from_LAST_DEBUG(LAST_DEBUG):
    """
    返回: verts(4x3), scene_id, next_node
    优先从 LAST_DEBUG['candidate'] 取；否则从 per_tet_snapshot + cand_simplex 恢复。
    scene_id：优先 candidate['best_scene']，否则默认为 0。
    """
    cand = LAST_DEBUG.get("candidate", {}) if isinstance(LAST_DEBUG, dict) else {}
    verts = cand.get("verts", None)
    if verts is None:
        # 备选：从 per_tet_snapshot 中按 cand_simplex 匹配
        sidx = LAST_DEBUG.get("cand_simplex", None)
        snap = LAST_DEBUG.get("per_tet_snapshot", None)
        if sidx is not None and isinstance(snap, list):
            for r in snap:
                if int(r.get("simplex_index", -1)) == int(sidx):
                    verts = r.get("verts", None)
                    if verts is not None:
                        break
    if verts is None:
        raise RuntimeError("无法从 LAST_DEBUG 提取单形顶点。请确保 LAST_DEBUG['candidate']['verts'] 或 per_tet_snapshot 可用。")
    verts = [tuple(map(float, v)) for v in verts]
    if len(verts) != 4:
        raise RuntimeError(f"单形顶点数量不是 4，实际得到 {len(verts)}")

    scene_id = cand.get("best_scene", 0)
    next_node = cand.get("x_ms_best_scene", None)
    if next_node is not None:
        next_node = tuple(map(float, next_node))

    return verts, int(scene_id), next_node

def _barycentric_grid_points(n_steps):
    """
    生成所有非负整数解 (i,j,k,l) 满足 i+j+k+l = n_steps。
    返回 lambdas 列表，每个 lam = (i/n, j/n, k/n, l/n)。
    """
    n = int(n_steps)
    lams = []
    for i in range(n+1):
        for j in range(n+1 - i):
            for k in range(n+1 - i - j):
                l = n - i - j - k
                lam = (i/n, j/n, k/n, l/n)
                lams.append(lam)
    return lams

def _eval_fverts_for_scene(verts, scene_id):
    """对给定场景 scene_id，在四个顶点上评估原函数 Q(v_j)。返回 list[4]"""
    vals = []
    for v in verts:
        vals.append(evaluate_Q_at(model_list[scene_id], first_stg_vars_list[scene_id], v, solver))
    return [float(x) for x in vals]

def visualize_obj_minus_As_on_simplex():
    # 依赖你主脚本中已有的对象
    needed = ["LAST_DEBUG", "model_list", "first_stg_vars_list", "solver"]
    for name in needed:
        if name not in globals():
            raise RuntimeError(f"需要全局变量 `{name}`，但当前环境中未找到。")

    verts, scene_id, next_node = _fetch_problem_simplex_from_LAST_DEBUG(LAST_DEBUG)
    v = np.array(verts, dtype=float)  # shape (4,3)

    # 计算顶点上的 f(v_j)（该 scene）
    fverts = _eval_fverts_for_scene(verts, scene_id)

    # 重心网格
    lambdas = _barycentric_grid_points(N_BARY)

    # 遍历网格：计算 x = Σ lam_j v_j，Q(x)，As = Σ lam_j f(v_j)，val = Q-As
    X, Y, Z, VAL = [], [], [], []
    for lam in lambdas:
        lam = np.array(lam, dtype=float)  # (4,)
        x = lam @ v  # (3,)
        # 评原函数（该 scene）
        Qx = evaluate_Q_at(model_list[scene_id], first_stg_vars_list[scene_id], tuple(map(float, x)), solver)
        As = float(np.dot(lam, fverts))
        val = float(Qx - As)
        X.append(x[0]); Y.append(x[1]); Z.append(x[2]); VAL.append(val)

    X = np.array(X); Y = np.array(Y); Z = np.array(Z); VAL = np.array(VAL)

    # 画 3D：单形 Mesh3d + 彩色散点（obj-As）+ next_node（若有）
    fig = go.Figure()

    # 单形的 4 个三角面
    I = [0, 0, 0, 1]
    J = [1, 1, 2, 2]
    K = [2, 3, 3, 3]

    fig.add_trace(go.Mesh3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        i=I, j=J, k=K,
        color="#888888", opacity=0.25, showscale=False,
        name="simplex"
    ))

    # 网格彩色点（obj-As）
    fig.add_trace(go.Scatter3d(
        x=X, y=Y, z=Z,
        mode="markers",
        marker=dict(size=3, color=VAL, colorscale="RdBu", colorbar=dict(title="obj - As")),
        name="grid (obj - As)"
    ))

    # next node（最小 ms 对应点）
    if next_node is not None:
        fig.add_trace(go.Scatter3d(
            x=[next_node[0]], y=[next_node[1]], z=[next_node[2]],
            mode="markers",
            marker=dict(size=7, color="green", symbol="diamond"),
            name="next node (min-ms)"
        ))

    # 也把四个顶点标出来
    fig.add_trace(go.Scatter3d(
        x=v[:,0], y=v[:,1], z=v[:,2],
        mode="markers+text",
        marker=dict(size=5, color="black"),
        text=[f"v{idx}" for idx in range(4)],
        textposition="top center",
        name="vertices"
    ))

    # 在 visualize_obj_minus_As_on_simplex() 里，绘图完成前加上这一段 —— 显示 too-close 的候选点
    too_close_pt = LAST_DEBUG.get("cand_point", None) if isinstance(LAST_DEBUG, dict) else None
    if too_close_pt is not None:
        too_close_pt = tuple(map(float, too_close_pt))
        fig.add_trace(go.Scatter3d(
            x=[too_close_pt[0]], y=[too_close_pt[1]], z=[too_close_pt[2]],
            mode="markers+text",
            marker=dict(size=7, symbol="x", color="red"),
            text=["too-close cand"],
            textposition="top center",
            name="too-close cand"
        ))


    fig.update_layout(
        title=f"obj - As over barycentric grid (scene={scene_id})",
        scene=dict(
            xaxis_title="Kp", yaxis_title="Ki", zaxis_title="Kd",
            aspectmode="cube"
        ),
        width=900, height=650,
        legend=dict(itemsizing="constant")
    )
    fig.show()

    # 返回数据，方便你做额外分析
    return {
        "verts": verts,
        "scene_id": scene_id,
        "next_node": next_node,
        "grid_points": np.column_stack([X, Y, Z]),
        "obj_minus_As": VAL,
        "fverts": fverts,
        "N_bary": N_BARY,
    }

viz_payload = visualize_obj_minus_As_on_simplex()